# 04 — Model Experiments

## 1. Objective

The goal of this notebook is to train and evaluate several classification models using the same cleaned data, train/validation split, and evaluation strategy.

Models tested in this notebook:

- Logistic Regression
- Random Forest
- XGBoost

Using the same data split allows a fair comparison between models.

## 2. Load Shared Data Split

In [11]:
import sys
sys.path.append("..")

from src.data_split import load_and_split_data

X_train, X_valid, y_train, y_valid = load_and_split_data()

print("X_train shape:", X_train.shape)
print("X_valid shape:", X_valid.shape)

print("\nTrain target distribution:")
print(y_train.value_counts(normalize=True).round(4))

X_train shape: (18819, 11)
X_valid shape: (4705, 11)

Train target distribution:
bank_account
0    0.8592
1    0.1408
Name: proportion, dtype: float64


## 3. Logistic Regression

Logistic Regression is used as a simple supervised classification model.

Unlike the DummyClassifier, this model learns from the input features, so we apply the shared preprocessing pipeline before training.

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

from src.preprocessing import build_scaled_preprocessor


logistic_model = Pipeline(
    steps=[
        ("preprocessor", build_scaled_preprocessor()),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

logistic_model.fit(X_train, y_train)

logistic_pred = logistic_model.predict(X_valid)

### 3.1 Evaluate Logistic Regression

We evaluate Logistic Regression using the same metrics as the baseline so the comparison remains consistent.

In [3]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    recall_score,
    confusion_matrix,
    classification_report
)

accuracy = accuracy_score(y_valid, logistic_pred)
error_rate = 1 - accuracy
macro_f1 = f1_score(y_valid, logistic_pred, average="macro")
yes_recall = recall_score(y_valid, logistic_pred, pos_label=1)
yes_f1 = f1_score(y_valid, logistic_pred, pos_label=1)

print("Accuracy:", round(accuracy, 4))
print("Error Rate:", round(error_rate, 4))
print("Macro F1:", round(macro_f1, 4))
print("Class 1 Recall:", round(yes_recall, 4))
print("Class 1 F1:", round(yes_f1, 4))

print("\nClassification Report:")
print(classification_report(y_valid, logistic_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, logistic_pred))

Accuracy: 0.8891
Error Rate: 0.1109
Macro F1: 0.708
Class 1 Recall: 0.361
Class 1 F1: 0.478

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      4043
           1       0.71      0.36      0.48       662

    accuracy                           0.89      4705
   macro avg       0.81      0.67      0.71      4705
weighted avg       0.88      0.89      0.87      4705


Confusion Matrix:
[[3944   99]
 [ 423  239]]


### Logistic Regression Interpretation

Logistic Regression clearly outperforms the DummyClassifier baseline.

- Accuracy improved to 88.91%.
- Error Rate decreased to 11.09%.
- Macro F1 increased substantially to 0.71.
- Class 1 Recall improved from 0.00 to 0.36.
- Class 1 F1 improved from 0.00 to 0.48.

The model now identifies a meaningful portion of respondents with bank accounts, although it still misses many positive cases.

This confirms that the input features contain useful predictive information beyond the majority-class baseline.

## 4. Random Forest

Random Forest is a tree-based ensemble model that can capture nonlinear relationships and interactions between features.

For this model, categorical features still need to be encoded, but numerical scaling is not required.

In [5]:
from sklearn.ensemble import RandomForestClassifier
from src.preprocessing import build_tree_preprocessor

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        (
            "model",
            RandomForestClassifier(
                n_estimators=200,
                random_state=42
            )
        )
    ]
)

random_forest_model.fit(X_train, y_train)

random_forest_pred = random_forest_model.predict(X_valid)

### 4.1 Evaluate Random Forest

We evaluate Random Forest using the same metrics as the previous models to keep the comparison consistent.

In [6]:
accuracy = accuracy_score(y_valid, random_forest_pred)
error_rate = 1 - accuracy
macro_f1 = f1_score(y_valid, random_forest_pred, average="macro")
yes_recall = recall_score(y_valid, random_forest_pred, pos_label=1)
yes_f1 = f1_score(y_valid, random_forest_pred, pos_label=1)

print("Accuracy:", round(accuracy, 4))
print("Error Rate:", round(error_rate, 4))
print("Macro F1:", round(macro_f1, 4))
print("Class 1 Recall:", round(yes_recall, 4))
print("Class 1 F1:", round(yes_f1, 4))

print("\nClassification Report:")
print(classification_report(y_valid, random_forest_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, random_forest_pred))

Accuracy: 0.8646
Error Rate: 0.1354
Macro F1: 0.6964
Class 1 Recall: 0.4275
Class 1 F1: 0.4705

Classification Report:
              precision    recall  f1-score   support

           0       0.91      0.94      0.92      4043
           1       0.52      0.43      0.47       662

    accuracy                           0.86      4705
   macro avg       0.72      0.68      0.70      4705
weighted avg       0.85      0.86      0.86      4705


Confusion Matrix:
[[3785  258]
 [ 379  283]]


### Random Forest Interpretation

Random Forest improves minority-class recall compared with Logistic Regression, identifying a larger share of respondents with bank accounts.

However, this improvement comes with more false positive predictions and lower overall accuracy.

- Accuracy: 86.46%
- Macro F1: 0.70
- Class 1 Recall: 0.43
- Class 1 F1: 0.47

Compared with Logistic Regression, Random Forest captures more positive cases but performs slightly worse overall under Macro F1.

## 5. XGBoost

XGBoost is a gradient-boosted tree model that can capture complex nonlinear relationships between features.

Like Random Forest, it does not require numerical scaling, but categorical variables still need to be encoded.

In [7]:
from xgboost import XGBClassifier
from src.preprocessing import build_tree_preprocessor

xgb_model = Pipeline(
    steps=[
        ("preprocessor", build_tree_preprocessor()),
        (
            "model",
            XGBClassifier(
                n_estimators=200,
                max_depth=4,
                learning_rate=0.05,
                random_state=42,
                eval_metric="logloss"
            )
        )
    ]
)

xgb_model.fit(X_train, y_train)

xgb_pred = xgb_model.predict(X_valid)

### 5.1 Evaluate XGBoost

We evaluate XGBoost using the same metrics as the previous models to keep the comparison consistent.

In [ ]:
accuracy = accuracy_score(y_valid, xgb_pred)
error_rate = 1 - accuracy
macro_f1 = f1_score(y_valid, xgb_pred, average="macro")
yes_recall = recall_score(y_valid, xgb_pred, pos_label=1)
yes_f1 = f1_score(y_valid, xgb_pred, pos_label=1)

print("Accuracy:", round(accuracy, 4))
print("Error Rate:", round(error_rate, 4))
print("Macro F1:", round(macro_f1, 4))
print("Class 1 Recall:", round(yes_recall, 4))
print("Class 1 F1:", round(yes_f1, 4))

print("\nClassification Report:")
print(classification_report(y_valid, xgb_pred, zero_division=0))

print("\nConfusion Matrix:")
print(confusion_matrix(y_valid, xgb_pred))

Accuracy: 0.8918
Error Rate: 0.1082
Macro F1: 0.8751
Class 1 Recall: 0.3565
Class 1 F1: 0.4811

Classification Report:
              precision    recall  f1-score   support

           0       0.90      0.98      0.94      4043
           1       0.74      0.36      0.48       662

    accuracy                           0.89      4705
   macro avg       0.82      0.67      0.71      4705
weighted avg       0.88      0.89      0.88      4705


Confusion Matrix:
[[3960   83]
 [ 426  236]]


### XGBoost Interpretation

XGBoost achieves the strongest overall performance among the tested models so far.

- Accuracy: 89.18%
- Error Rate: 10.82%
- Macro F1: 0.71
- Class 1 Recall: 0.36
- Class 1 F1: 0.48

XGBoost slightly outperforms Logistic Regression in overall balanced performance, while Random Forest still achieves the highest minority-class recall.

This shows a trade-off between overall model quality and the ability to identify more positive cases.

## 6. Cross-Validation Evaluation

To obtain a more robust comparison between the supervised models, we evaluate them using 5-fold stratified cross-validation.

Stratification preserves the class distribution across folds, which is important because the target is imbalanced.

The main evaluation metric is Macro F1.

In [12]:
from sklearn.model_selection import StratifiedKFold, cross_validate

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

scoring = {
    "accuracy": "accuracy",
    "macro_f1": "f1_macro",
    "class1_recall": "recall",
    "class1_f1": "f1"
}

models = {
    "Logistic Regression": logistic_model,
    "Random Forest": random_forest_model,
    "XGBoost": xgb_model
}

In [13]:
cv_results = []

for model_name, model in models.items():
    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_results.append({
        "Model": model_name,
        "CV Accuracy": scores["test_accuracy"].mean(),
        "CV Macro F1": scores["test_macro_f1"].mean(),
        "CV Class 1 Recall": scores["test_class1_recall"].mean(),
        "CV Class 1 F1": scores["test_class1_f1"].mean()
    })

cv_results = pd.DataFrame(cv_results)

cv_results.round(4)

,Model,CV Accuracy,CV Macro F1,CV Class 1 Recall,CV Class 1 F1
0,Logistic Regression,0.8840,0.6920,0.3362,0.4488
1,Random Forest,0.8613,0.6769,0.3758,0.4329
2,XGBoost,0.8878,0.6964,0.3332,0.4553


### Cross-Validation Interpretation

- XGBoost achieves the highest mean CV Macro F1 score.
- Logistic Regression performs very similarly to XGBoost.
- Random Forest achieves the highest Class 1 Recall but lower overall Macro F1.
- Based on cross-validation, XGBoost remains the current best model.

## 7. Model Comparison

The models are compared using both the fixed validation split and 5-fold stratified cross-validation.

Cross-validation is used as the main basis for model selection because it provides a more robust estimate of model performance.

The primary metric is Macro F1.

In [ ]:
import pandas as pd

results = pd.DataFrame({
    "Model": [
        "Dummy Classifier",
        "Logistic Regression",
        "Random Forest",
        "XGBoost"
    ],
    "Accuracy": [
        0.8593,
        0.8891,
        0.8646,
        0.8918
    ],
    "Error Rate": [
        0.1407,
        0.1109,
        0.1354,
        0.1082
    ],
    "Macro F1": [
        0.4622,
        0.7080,
        0.6964,
        0.7104
    ],
    "Class 1 Recall": [
        0.0000,
        0.3610,
        0.4275,
        0.3565
    ],
    "Class 1 F1": [
        0.0000,
        0.4780,
        0.4705,
        0.4811
    ]
})

results

,Model,CV Accuracy,CV Macro F1,CV Class 1 Recall,CV Class 1 F1
0,Logistic Regression,0.8840,0.6920,0.3362,0.4488
1,Random Forest,0.8613,0.6769,0.3758,0.4329
2,XGBoost,0.8878,0.6964,0.3332,0.4553


In [15]:
cv_results.round(4)

,Model,CV Accuracy,CV Macro F1,CV Class 1 Recall,CV Class 1 F1
0,Logistic Regression,0.8840,0.6920,0.3362,0.4488
1,Random Forest,0.8613,0.6769,0.3758,0.4329
2,XGBoost,0.8878,0.6964,0.3332,0.4553


### Model Comparison Interpretation

- XGBoost achieves the highest mean CV Macro F1 score: 0.6964.
- Logistic Regression performs very similarly with a CV Macro F1 of 0.6920.
- Random Forest achieves the highest Class 1 Recall but a lower Macro F1.
- The cross-validation results are consistent with the fixed validation results.

### Current Best Model

Based on 5-fold stratified cross-validation and Macro F1, XGBoost remains the current best model.